# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors
Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities via their `@id` fields as per best practice.

**Dataset Source**  
Croissant JSON-LD: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

*Fields, columns, and record sets below are referenced by their `@id`.*


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview
List all available record sets with their `@id`. For each, show their field `@id` and names for inspection.

In [ ]:
# Get all record sets in the dataset
record_sets = dataset.metadata.record_sets
print("Record sets in this dataset:")

for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}, name: {rs.get('name', '--')}\n    Fields:")
    for field in rs.get('field', []):
        print(f"    - Field @id: {field['@id']}   Name: {field.get('name', '--')}")


## 3. Data Extraction
Load data for each record set, referencing them using their `@id`.

You can pick the main record set (`@id`) of interest for your analysis.

In [ ]:
# Collect all record set IDs
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    # Each record is a dict of field `@id`: value
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet {rs_id}")
        print(f"Fields (column @id): {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(2))
    else:
        print(f"RecordSet {rs_id} is empty or not directly downloadable.")


## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (referenced by its `@id`) for filtering, normalization and basic grouping. Please set the variables below (`main_record_set`, `numeric_field_id`, `group_field_id`) according to the output above for the column you want to explore.

In [ ]:
# ---- USER: Set the desired RecordSet and field `@id`s for analysis ---- #
main_record_set = next(iter(dataframes))  # Example: choose the first loaded
numeric_field_id = None
group_field_id = None

# Suggest field selection by showing all columns
print(f"Available columns in {main_record_set}:")
print(dataframes[main_record_set].columns.tolist())

if numeric_field_id is None:
    # Try to auto-detect a numeric field:
    df = dataframes[main_record_set]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    print(f"Auto-selected numeric field: {numeric_field_id}")
else:
    df = dataframes[main_record_set]

if group_field_id is None:
    # Try to auto-detect a categorical field for grouping
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    print(f"Auto-selected group field: {group_field_id}")

# Filter for numeric_field_id > threshold
threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field if available
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and the average per group. (Requires `matplotlib` or `seaborn`.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Barplot of grouped means
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library for Fair, Accessible, and Interoperable dataset analysis, referencing all data structures by their `@id` fields as required by Croissant best practices.

You can adapt the field `@id`s (see cell output above for the options) to analyze other variables or combinations as desired.